# Support Vector Machines

Implement a linear soft-margin SVM from scratch using the hinge-loss + L2 primal objective, trained with (sub)gradient descent. Validate against scikit-learn and visualise the decision boundary and margin.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — see the repo root for defaults.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## The soft-margin SVM objective

For labels \(y_i \in \{-1, +1\}\) and score \(f(x) = w^\top x + b\), the **primal soft-margin objective** is:

$$\mathcal{L}(w, b) = \frac{1}{n} \sum_i \max(0,\, 1 - y_i (w^\top x_i + b)) + \frac{\lambda}{2} \|w\|_2^2$$

The first term is the **hinge loss** — it is zero for correctly classified points with margin \(\geq 1\), and grows linearly otherwise.  The second term is L2 regularisation that keeps \(\|w\|\) small (equivalently, the geometric margin \(2/\|w\|\) large).

The relationship between this primal form and the usual \(C\)-parameterised SVM is \(\lambda = 1/(2nC)\) when the hinge sum is over all \(n\) examples.

In [2]:
import torch

torch.manual_seed(42)

N = 200  # total samples
# Two 2-D Gaussian blobs, slightly overlapping
X_pos = torch.randn(N // 2, 2) + torch.tensor([1.5, 1.5])
X_neg = torch.randn(N // 2, 2) + torch.tensor([-1.5, -1.5])

X = torch.cat([X_pos, X_neg], dim=0).to(device)          # (200, 2)
y = torch.cat([torch.ones(N // 2), -torch.ones(N // 2)]).to(device)  # +1 / -1

# Shuffle
perm = torch.randperm(N, device=device)
X, y = X[perm], y[perm]

# Train / test split (80 / 20)
split = int(0.8 * N)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Positive labels: {(y_train == 1).sum().item()}, Negative: {(y_train == -1).sum().item()}")

Train: torch.Size([160, 2]), Test: torch.Size([40, 2])
Positive labels: 82, Negative: 78


## Linear SVM from scratch

We minimise \(\mathcal{L}\) with subgradient descent.  The subgradient at a point where \(y_i f(x_i) < 1\) (hinge active) is \(-y_i x_i\) (for \(w\)) and \(-y_i\) (for \(b\)); otherwise it is zero.

In [3]:
class LinearSVMScratch:
    """Soft-margin linear SVM via subgradient descent on the primal objective."""

    def __init__(self, lam: float = 0.01, lr: float = 0.01, n_iters: int = 2000):
        self.lam = lam
        self.lr = lr
        self.n_iters = n_iters
        self.w: torch.Tensor | None = None
        self.b: torch.Tensor | None = None
        self.losses: list[float] = []

    def fit(self, X: torch.Tensor, y: torch.Tensor) -> "LinearSVMScratch":
        n, d = X.shape
        w = torch.zeros(d, device=X.device, requires_grad=False)
        b = torch.zeros(1, device=X.device, requires_grad=False)

        for _ in range(self.n_iters):
            scores = X @ w + b          # (n,)
            margins = y * scores        # (n,)
            active = margins < 1.0      # boolean mask — hinge is active

            # Mean hinge loss + L2 penalty
            hinge = torch.clamp(1.0 - margins, min=0.0).mean()
            loss = hinge + (self.lam / 2.0) * (w @ w)
            self.losses.append(loss.item())

            # Subgradient
            dw = self.lam * w - (y[active, None] * X[active]).mean(dim=0) if active.any() else self.lam * w
            db = -y[active].mean() if active.any() else torch.zeros(1, device=X.device)

            w = w - self.lr * dw
            b = b - self.lr * db

        self.w = w
        self.b = b
        return self

    def decision_function(self, X: torch.Tensor) -> torch.Tensor:
        return X @ self.w + self.b

    def predict(self, X: torch.Tensor) -> torch.Tensor:
        return torch.sign(self.decision_function(X))

    def accuracy(self, X: torch.Tensor, y: torch.Tensor) -> float:
        return (self.predict(X) == y).float().mean().item()


# Train
svm_scratch = LinearSVMScratch(lam=0.01, lr=0.05, n_iters=3000)
svm_scratch.fit(X_train, y_train)

train_acc = svm_scratch.accuracy(X_train, y_train)
test_acc  = svm_scratch.accuracy(X_test,  y_test)
print(f"From-scratch SVM  —  train acc: {train_acc:.4f}  |  test acc: {test_acc:.4f}")

From-scratch SVM  —  train acc: 0.9812  |  test acc: 0.9750


In [4]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(svm_scratch.losses)
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("Subgradient descent: training loss")
ax.set_yscale("log")
fig.tight_layout()
plt.savefig("svm_loss_curve.png", dpi=80)
plt.show()
print("Loss curve saved.")

Loss curve saved.


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_71433/1550183943.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sklearn baseline

`sklearn.svm.LinearSVC` solves the same primal objective with an optimised solver (LIBLINEAR).  We fit it here for accuracy comparison.

In [5]:
import numpy as np
from sklearn.svm import LinearSVC

X_train_np = X_train.cpu().numpy()
X_test_np  = X_test.cpu().numpy()
y_train_np = y_train.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

# C=1/(2*n*lam) to match our lambda=0.01 (n_train=160)
lam_scratch = 0.01
n_train = X_train_np.shape[0]
C_equiv = 1.0 / (2.0 * n_train * lam_scratch)

sk_svm = LinearSVC(C=C_equiv, max_iter=10_000, random_state=42)
sk_svm.fit(X_train_np, y_train_np)

sk_train_acc = sk_svm.score(X_train_np, y_train_np)
sk_test_acc  = sk_svm.score(X_test_np,  y_test_np)
print(f"sklearn LinearSVC  —  train acc: {sk_train_acc:.4f}  |  test acc: {sk_test_acc:.4f}")

sklearn LinearSVC  —  train acc: 0.9750  |  test acc: 0.9750


## Validation

We assert that our from-scratch accuracy is within 5 percentage points of sklearn on both train and test sets.

In [6]:
tol = 0.05

assert abs(train_acc - sk_train_acc) <= tol, (
    f"Train accuracy gap too large: scratch={train_acc:.4f}, sklearn={sk_train_acc:.4f}"
)
assert abs(test_acc - sk_test_acc) <= tol, (
    f"Test accuracy gap too large: scratch={test_acc:.4f}, sklearn={sk_test_acc:.4f}"
)

print(f"PASS: scratch vs sklearn gap ≤ {tol}")
print(f"  scratch train/test: {train_acc:.4f} / {test_acc:.4f}")
print(f"  sklearn  train/test: {sk_train_acc:.4f} / {sk_test_acc:.4f}")

PASS: scratch vs sklearn gap ≤ 0.05
  scratch train/test: 0.9812 / 0.9750
  sklearn  train/test: 0.9750 / 0.9750


## Decision boundary and margin

The decision boundary is \(w^\top x + b = 0\).  The two margin hyperplanes are \(w^\top x + b = \pm 1\).  Points with raw margin \(|y_i f(x_i)| \leq 1\) are **support vectors** (they sit inside or on the margin street).

In [7]:
def plot_svm_boundary(ax, X_np: np.ndarray, y_np: np.ndarray,
                      w: np.ndarray, b: float, title: str) -> None:
    """Draw scatter + decision boundary + margins + support-vector circles."""
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5

    pos_mask = y_np == 1
    ax.scatter(X_np[pos_mask, 0],  X_np[pos_mask, 1],  c="#2196F3", label="+1", alpha=0.6, s=25)
    ax.scatter(X_np[~pos_mask, 0], X_np[~pos_mask, 1], c="#F44336", label="-1", alpha=0.6, s=25)

    # Decision boundary: w[0]*x + w[1]*y + b = offset  =>  y = (offset - w[0]*x - b) / w[1]
    xs = np.linspace(x_min, x_max, 200)
    if abs(w[1]) > 1e-9:
        boundary = (-w[0] * xs - b) / w[1]
        margin_p = (-w[0] * xs - b + 1) / w[1]
        margin_n = (-w[0] * xs - b - 1) / w[1]
        ax.plot(xs, boundary, "k-",  lw=2,   label="boundary")
        ax.plot(xs, margin_p, "k--", lw=1.2, label="margin +1")
        ax.plot(xs, margin_n, "k--", lw=1.2, label="margin -1")

    # Support vectors: raw margin |y*f(x)| <= 1
    scores   = X_np @ w + b
    margins_ = y_np * scores
    sv_mask  = margins_ <= 1.0
    ax.scatter(X_np[sv_mask, 0], X_np[sv_mask, 1],
               s=120, facecolors="none", edgecolors="gold", linewidths=1.5, label="support vectors")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(title)
    ax.legend(fontsize=7)


w_scratch = svm_scratch.w.cpu().numpy()
b_scratch = svm_scratch.b.cpu().numpy().item()

X_all_np = X.cpu().numpy()
y_all_np = y.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_svm_boundary(axes[0], X_all_np, y_all_np, w_scratch, b_scratch,
                  "From-scratch SVM (hinge + L2 subgradient)")
plot_svm_boundary(axes[1], X_all_np, y_all_np,
                  sk_svm.coef_[0], sk_svm.intercept_[0],
                  "sklearn LinearSVC")

fig.tight_layout()
plt.savefig("svm_boundaries.png", dpi=80)
plt.show()

margin_width_scratch = 2.0 / np.linalg.norm(w_scratch)
margin_width_sk      = 2.0 / np.linalg.norm(sk_svm.coef_[0])
print(f"Geometric margin width — scratch: {margin_width_scratch:.4f} | sklearn: {margin_width_sk:.4f}")

n_sv_scratch = int(((y_all_np * (X_all_np @ w_scratch + b_scratch)) <= 1).sum())
print(f"Support vectors (margin ≤ 1): {n_sv_scratch}")

Geometric margin width — scratch: 0.7746 | sklearn: 1.9143
Support vectors (margin ≤ 1): 9


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_71433/52140527.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Effect of C on the margin

Large \(C\) penalises hinge violations heavily → narrow margin (may overfit).  Small \(C\) tolerates violations → wide margin (more regularised).

In [8]:
lam_values = [0.001, 0.01, 0.1, 1.0]   # increasing lam = decreasing C
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, lam in zip(axes, lam_values):
    m = LinearSVMScratch(lam=lam, lr=0.05, n_iters=3000)
    m.fit(X_train, y_train)
    w_np = m.w.cpu().numpy()
    b_np = m.b.cpu().numpy().item()
    mw   = 2.0 / (np.linalg.norm(w_np) + 1e-9)
    acc  = m.accuracy(X_test, y_test)
    plot_svm_boundary(ax, X_all_np, y_all_np, w_np, b_np,
                      f"λ={lam}  margin={mw:.2f}  acc={acc:.2f}")

fig.tight_layout()
plt.savefig("svm_C_sweep.png", dpi=80)
plt.show()
print("C sweep saved.")

C sweep saved.


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_71433/3191307062.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idiomatic sklearn

`sklearn.svm.SVC(kernel='linear')` solves the *dual* formulation and can be extended to kernel SVMs via a single `kernel=` argument change.

In [9]:
from sklearn.svm import SVC

sk_svc = SVC(kernel="linear", C=1.0, random_state=42)
sk_svc.fit(X_train_np, y_train_np)

print(f"SVC(kernel='linear') — train: {sk_svc.score(X_train_np, y_train_np):.4f}"
      f"  test: {sk_svc.score(X_test_np, y_test_np):.4f}")
print(f"Number of support vectors per class: {sk_svc.n_support_}")

SVC(kernel='linear') — train: 0.9812  test: 0.9750
Number of support vectors per class: [6 6]


## The kernel trick (conceptually)

A kernel \(K(x, x') = \phi(x)^\top \phi(x')\) computes the dot product that would result after mapping inputs into a (possibly infinite-dimensional) feature space \(\phi\).  The SVM dual objective depends on inputs *only* through these dot products, so we can swap in \(K\) without ever materialising \(\phi(x)\).

Common kernels:
- **Linear**: \(K(x,x') = x^\top x'\) — equivalent to the model above.
- **RBF/Gaussian**: \(K(x,x') = \exp(-\gamma \|x-x'\|^2)\) — infinite-dimensional feature map, non-linear boundary.
- **Polynomial**: \(K(x,x') = (x^\top x' + c)^d\).

Kernel SVMs are powerful for small/medium datasets with non-linear structure, but scale as \(O(n^2)\) in memory and \(O(n_{sv} \cdot d)\) in prediction cost.

## Takeaways

- A **linear SVM** finds the maximum-margin hyperplane; the **soft-margin** variant (hinge loss + L2 penalty) handles non-separable data.
- Only **support vectors** (points with margin \(\leq 1\)) influence the solution — SVMs are inherently sparse in data space.
- The **hinge loss** is piecewise linear with a subgradient; plain subgradient descent converges, though more slowly than primal/dual solvers.
- \(C\) (or equivalently \(\lambda\)) balances margin width against training error; tune it via cross-validation.
- The **kernel trick** extends SVMs to non-linear boundaries without explicit feature construction.
- In modern practice, SVMs excel for **high-dimensional sparse** inputs (text) and **small-to-medium** tabular datasets where a large margin prior is appropriate.